# HuggingFace 实战
https://huggingface.co/
> 在人工智能开发领域，Hugging Face 已成为一个不可或缺的平台，它让开发者能够轻松调用数百个预训练模型，快速实现各种NLP和CV任务。
> 1. NLP（Natural Language Processing）: 自然语言处理
> 2. CV（Computer Vision）：计算机视觉

# 1. 基础概念
## 1. 什么是HuggingFace
Hugging Face 是一个用于分享和使用机器学习模型的平台，提供了一个开放的模型共享社区。它不仅提供丰富的预训练模型、强大的工具库，还构建了一个活跃的技术社区。与传统科技公司的封闭产品不同，Hugging Face采用去中心化的运作模式，研究人员、企业和独立开发者共同为一个共享的基础设施贡献力量。

## 2. 大模型分类
![image.png](./img/mxfl.png)

## 3. 核心组件
Hugging Face生态系统主要由以下几个核心组件构成：
1. Transformers库：提供了大量主流NLP模型【自然语言处理(Natural Language Processing)】（如BERT、GPT、T5等）的统一接口
2. Datasets库：提供标准化的数据集接口，支持在线加载、缓存、切片等操作
3. Tokenizers库：提供高性能的分词器，支持多种分词算法
4. Hub平台：作为模型仓库，用户可以上传、下载、版本化模型

# 2. Transformers 库用法
## 1. Pipeline API
Pipeline是Hugging Face中最简单且强大的API，它将复杂的模型调用流程简化为端到端的处理过程。
一个完整的模型应用通常包含三个关键组件：
- 分词器(Tokenizer)
- 模型本身(Model)
- 后处理器(Post-processor)
Pipeline技术将这三大组件有机整合，形成统一接口

## 2. 安装依赖

In [1]:
# 安装核心库
!pip install transformers datasets tokenizers huggingface_hub
# 安装开发工具
# !pip install huggingface_hub

/bin/bash: line 1: pip: command not found


## 3. 验证安装

In [1]:
import transformers
print(f"Transformers版本: {transformers.__version__}")

import datasets
print(f"Datasets版本: {datasets.__version__}")

/usr/local/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers版本: 4.57.1
Datasets版本: 4.3.0


## 4. 案例 - 情感分类
### 1. 基本用法

In [3]:
from transformers import pipeline

# 方法1：明确指定模型名称； 如果不指定模型的名字，会下载一个默认模型； distilbert/distilbert-base-uncased-finetuned-sst-2-english
classifier = pipeline(
    'sentiment-analysis',
    model='distilbert/distilbert-base-uncased-finetuned-sst-2-english'
)

result = classifier("I love using Hugging Face libraries!")
print(result)

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9981693029403687}]


### 2. 中文情感分析模型

In [5]:
# 设置镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from transformers import pipeline

# 如果需要中文情感分析
classifier = pipeline(
    'sentiment-analysis',
    model='uer/roberta-base-finetuned-jd-binary-chinese'
)

result = classifier("我很讨厌编程")
print(result)


Device set to use cuda:0


[{'label': 'negative (stars 1, 2 and 3)', 'score': 0.8346632719039917}]


### 3. 模型缓存位置

> 下载的模型会保存到本地位置：
1. 默认位置：~/.cache/huggingface/hub（Linux/macOS）或 C:\Users\<用户名>\.cache\huggingface\hub（Windows）。
2. 改位置：用环境变量 HF_HOME（改全部HF缓存[模型和数据集位置都修改]）或 TRANSFORMERS_CACHE（只改Transformers模型缓存）。

### 4. 多文本处理

In [6]:
from transformers import pipeline
# 设置镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

classifier = pipeline(
    'sentiment-analysis',
    model='distilbert/distilbert-base-uncased-finetuned-sst-2-english'
)

# 批量处理
texts = [
    "I love using Hugging Face libraries!",
    "This is terrible.",
    "Not bad, could be better."
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"Text: {text}")
    print(f"Result: {result}")
    print("-" * 50)


Device set to use cuda:0


Text: I love using Hugging Face libraries!
Result: {'label': 'POSITIVE', 'score': 0.9981693029403687}
--------------------------------------------------
Text: This is terrible.
Result: {'label': 'NEGATIVE', 'score': 0.9996345043182373}
--------------------------------------------------
Text: Not bad, could be better.
Result: {'label': 'POSITIVE', 'score': 0.9797654747962952}
--------------------------------------------------


### 5. 返回所有标签概率

In [8]:
from transformers import pipeline

classifier = pipeline(
    'sentiment-analysis',
    model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',
    top_k=None
    # return_all_scores=True  # 返回所有标签的概率
)

result = classifier("I love using Hugging Face libraries!")
print(result)

Device set to use cuda:0


[[{'label': 'POSITIVE', 'score': 0.9981693029403687}, {'label': 'NEGATIVE', 'score': 0.001830678666010499}]]


### 6. 手动下载模型
https://github.com/huggingface/huggingface_hub/blob/main/i18n/README_cn.md

In [ ]:
# 必须安装 huggingface_hub 依赖，才可以使用API进行模型下载
!pip install huggingface_hub

#### 1. 下载单个文件
下载单个文件, 使用 hf_hub_download + 指定文件名即可


In [9]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="distilbert/distilbert-base-uncased-finetuned-sst-2-english", 
    filename="config.json")
print(model_path)

/root/.cache/huggingface/hub/models--distilbert--distilbert-base-uncased-finetuned-sst-2-english/snapshots/714eb0fa89d2f80546fda750413ed43d93601a13/config.json


#### 2. 下载整个模型
下载整个模型, 使用 snapshot_download 即可；
snapshot_download(指定模型名字)： 这是最简单用法

In [ ]:
from huggingface_hub import snapshot_download

# 1. 下载到默认位置
model_path = snapshot_download("distilbert/distilbert-base-uncased-finetuned-sst-2-english")
print(model_path)

In [13]:
from huggingface_hub import snapshot_download

# 2. 下载到指定位置
model_path = snapshot_download(
    repo_id="distilbert/distilbert-base-uncased-finetuned-sst-2-english", 
    local_dir="./model_cache/distilbert-base-uncased-finetuned-sst-2-english",
    local_dir_use_symlinks=False  # 不使用符号链接，直接复制文件
)
print(model_path)

Fetching 17 files: 100%|██████████| 17/17 [00:01<00:00, 14.24it/s]

/workspace/model_cache/distilbert-base-uncased-finetuned-sst-2-english


### 7. 加载本地模型
> 我们使用 hf API 下载模型缓存到本地以后，可以随时的加载离线模型
> 用法如下：

#### 1. 下载模型

In [14]:
from huggingface_hub import snapshot_download

local_model_path = "./model_cache/finbert"
model_path = snapshot_download(
    repo_id="ProsusAI/finbert",
    local_dir=local_model_path,
    local_dir_use_symlinks=False # 不使用符号链接，直接复制文件
)
print(f"模型已下载到: {model_path}")

Fetching 9 files: 100%|██████████| 9/9 [00:44<00:00,  4.93s/it]

模型已下载到: /workspace/model_cache/finbert


#### 2. 加载模型

In [15]:
from transformers import pipeline
import os

# 使用本地模型
local_model_path = "./model_cache/finbert"

if os.path.exists(local_model_path):
    classifier = pipeline(
        'sentiment-analysis',
        model=local_model_path,
        tokenizer=local_model_path,
        return_all_scores=True
    )
    
    result = classifier("I love using Hugging Face libraries!")
    print(result)
else:
    print("本地模型不存在，请先下载模型")

Device set to use cuda:0


[[{'label': 'positive', 'score': 0.06913752853870392}, {'label': 'negative', 'score': 0.014439063146710396}, {'label': 'neutral', 'score': 0.9164233803749084}]]
